In [1]:
import pennylane as qml
from pennylane import numpy as np


In [5]:
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

print(X.shape)  # (150, 4)

(150, 4)


In [6]:
import random

X_min = X.min(axis=0)   # min of each column
X_max = X.max(axis=0)   # max of each column

X = (X - X_min) / (X_max - X_min)

X = X * (2*np.pi) - np.pi   # [-π, π]



# Example lists (columns)
A = X
B = y

# Combine into rows
rows = list(zip(A, B))

# Shuffle rows
random.shuffle(rows)

# Unzip back into columns
A_shuf, B_shuf = zip(*rows)

# (optional) convert back to lists
X= list(A_shuf)
y = list(B_shuf)



X_train = X[:100]
y_train = y[:100]

X_test = X[100:]
y_test = y[100:]


In [7]:
dev = qml.device("default.qubit", wires = 3)

In [23]:
@qml.qnode(dev)
def circuit(params, x):
    
    qml.RX(x[0], 0)
    qml.RX(x[1], 1)
    qml.RX(x[2], 2)

    qml.RY(params[0], 0)
    qml.RY(params[1], 1)
    qml.RY(params[2], 2)
    qml.CNOT([2, 1])
    qml.CNOT([1, 0])
    qml.RZ(params[3], 0)
    qml.RZ(params[4], 1)
    qml.RZ(params[5], 2)


    qml.CNOT([0, 1])
    qml.CNOT([1, 2])


    qml.RX(x[1], 0)    
    qml.RX(x[2], 1)
    qml.RX(x[3], 2)

    qml.RY(params[6], 0)
    qml.RY(params[7], 1)
    qml.RY(params[8], 2)
    qml.CNOT([2, 1])
    qml.CNOT([1, 0])
    qml.RZ(params[9], 0)
    qml.RZ(params[10], 1)
    qml.RZ(params[11], 2)


    qml.CNOT([0, 1])
    qml.CNOT([1, 2])


    qml.RX(x[0] + x[1], 0)
    qml.RX(x[1] + x[2], 1)
    qml.RX(x[2] + x[3], 2)

    qml.RY(params[12], 0)
    qml.RY(params[13], 1)
    qml.RY(params[14], 2)
    qml.CNOT([1, 0])
    qml.CNOT([2, 1])
    qml.RZ(params[15], 0)
    qml.RZ(params[16], 1)
    qml.RZ(params[17], 2)

    qml.CNOT([0, 1])
    qml.CNOT([1, 2])

    qml.RX(x[0] + x[3], 0)
    qml.RX(x[1] + x[2], 1)
    qml.RX(x[1] + x[2], 2)


        
    return qml.probs(wires = [0, 1, 2])

In [24]:
def cost(params, X, y):
    loss = 0

    for x, y_true in zip(X, y):
        probs = circuit(params, x)

        p0 = probs[0] + probs[1]
        p1 = probs[2] + probs[3]
        p2 = probs[4] + probs[5]

        p = [p0, p1, p2][y_true]

        loss -= np.log(p + 1e-10)
        loss += 0.1 * probs[6] + probs[7]

    return loss / len(X)

In [11]:
def compute_accuracy(params, X, y):
    correct = 0

    for x, y_true in zip(X, y):
        probs = circuit(params, x)

        p0 = probs[0] + probs[1]
        p1 = probs[2] + probs[3]
        p2 = probs[4] + probs[5]

        y_pred = np.argmax([p0, p1, p2])

        if y_pred == y_true:
            correct += 1

    return correct / len(X)

In [25]:
import random

n = 80

np.random.seed(n)
random.seed(n)   # <-- this was missing

params = np.random.rand(18, requires_grad=True) #* (2*np.pi) - np.pi   # [-π, π]

In [26]:
opt = qml.AdamOptimizer(stepsize=0.05)

steps = 50

for i in range(steps):
    params = opt.step(lambda p: cost(p, X_train, y_train), params)

    if i % 2 == 0:
        train_loss = cost(params, X_train, y_train)
        train_acc = compute_accuracy(params, X_train, y_train)
        test_acc = compute_accuracy(params, X_test, y_test)

        print(f"Step {i:3d} | Loss: {train_loss:.6f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

Step   0 | Loss: 1.553745 | Train Acc: 0.4400 | Test Acc: 0.2600
Step   2 | Loss: 1.432660 | Train Acc: 0.5400 | Test Acc: 0.4000
Step   4 | Loss: 1.337584 | Train Acc: 0.5600 | Test Acc: 0.4200
Step   6 | Loss: 1.271961 | Train Acc: 0.5900 | Test Acc: 0.4600
Step   8 | Loss: 1.225119 | Train Acc: 0.6000 | Test Acc: 0.5400
Step  10 | Loss: 1.184895 | Train Acc: 0.5800 | Test Acc: 0.5400
Step  12 | Loss: 1.146556 | Train Acc: 0.6300 | Test Acc: 0.5400
Step  14 | Loss: 1.109904 | Train Acc: 0.6700 | Test Acc: 0.5600
Step  16 | Loss: 1.074213 | Train Acc: 0.6800 | Test Acc: 0.5600
Step  18 | Loss: 1.038487 | Train Acc: 0.6800 | Test Acc: 0.5600
Step  20 | Loss: 1.002356 | Train Acc: 0.6900 | Test Acc: 0.5800
Step  22 | Loss: 0.966471 | Train Acc: 0.7000 | Test Acc: 0.5800
Step  24 | Loss: 0.932889 | Train Acc: 0.7500 | Test Acc: 0.5600
Step  26 | Loss: 0.902876 | Train Acc: 0.7700 | Test Acc: 0.6000
Step  28 | Loss: 0.876588 | Train Acc: 0.7800 | Test Acc: 0.6000
Step  30 | Loss: 0.855007

In [22]:
unused_vals = []

for x in X_test:
    probs = circuit(params, x)
    unused_vals.append(probs[6] + probs[7])

print("Avg unused:", np.mean(unused_vals))

Avg unused: 0.14820470715757708
